In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

**Trying on with background subtraction**

In [ ]:
reference_image=r"C:\ALL PROJECTS\MCCB-Defect-Detection\masterImages\masterP13P_mccb.png"
test_image=r"C:\ALL PROJECTS\MCCB-Defect-Detection\images\mccb(T).png"
img=cv2.imread(reference_image)
print(img.shape)

In [ ]:
reference=cv2.imread(reference_image,0)
print(reference.shape)




In [ ]:
test=cv2.imread(test_image,0)
print(test.shape)

In [ ]:
test=cv2.resize(test,(reference.shape[1],reference.shape[0]))
print(test.shape)

In [ ]:
test = cv2.GaussianBlur(test, (5,5), 0)
reference = cv2.GaussianBlur(reference, (5,5), 0)


In [ ]:
fig,(ax1,ax2)=plt.subplots(1,2,figsize=(10,5))
ax1.imshow(reference,cmap='gray')
ax1.set_title('Reference')
ax2.imshow(test,cmap='gray')
ax2.set_title('Test')
ax1.axis('off')
ax2.axis('off')
plt.show()

In [ ]:
diff1=cv2.absdiff(reference,test)
diff0=cv2.absdiff(reference,test)
_,mask=cv2.threshold(diff1,30,255,cv2.THRESH_BINARY)

In [ ]:
fig,(ax1,ax2)=plt.subplots(1,2,figsize=(12,6))
ax1.imshow(diff0,cmap='gray')
ax2.imshow(mask,cmap='gray')
ax1.set_title("before background subtraction")
ax2.set_title("after background subtraction")
ax1.axis('off')
ax2.axis('off')
plt.show()


In [ ]:
kernel=np.ones((3,3),np.uint8)
clean_mask=cv2.medianBlur(mask, 5)

In [ ]:
test_color=cv2.cvtColor(test,cv2.COLOR_GRAY2BGR)
test_color[clean_mask==255]=(0,255,0)
plt.imshow(test_color)

In [ ]:
contours, _ = cv2.findContours(clean_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
test_color=cv2.cvtColor(test,cv2.COLOR_GRAY2BGR)
for cnt in contours:
    x,y,w,h=cv2.boundingRect(cnt)
    cv2.rectangle(test_color,(x,y),(x+w,y+h),(0,0,255),2)
fig,(ax1,ax2,ax3)=plt.subplots(1,3,figsize=(12,6))
ax1.imshow(reference,cmap='gray')
ax2.imshow(test,cmap='gray')
ax3.imshow(test_color)
ax1.set_title("reference")
ax2.set_title("test")
ax3.set_title("contours")
ax1.axis('off')
ax2.axis('off')
ax3.axis('off')
plt.show()

In [ ]:
plt.imshow(test_color)
plt.show()

**Gradient difference method**

In [ ]:
master_path=r"C:\ALL PROJECTS\MCCB-Defect-Detection\images\mccb(R).png"
test_path=r"C:\ALL PROJECTS\MCCB-Defect-Detection\images\mccb(T).png"

In [ ]:
#Reading image
master_image=cv2.imread(master_path)
test_image=cv2.imread(test_path)
print(master_image.shape)
print(test_image.shape)


In [ ]:
#Resize (Actually ignored in the main code since it is a controlled environment)
master_image=cv2.resize(master_image,(test_image.shape[1],test_image.shape[0]))

In [ ]:
#Grayscaling
master_image_gray=cv2.cvtColor(master_image,cv2.COLOR_BGR2GRAY)
test_image_gray=cv2.cvtColor(test_image,cv2.COLOR_BGR2GRAY)

In [ ]:
master_blur = cv2.GaussianBlur(master_image_gray, (5,5), 0)
test_blur = cv2.GaussianBlur(test_image_gray, (5,5), 0)

In [ ]:
#CLAHE(Contrast Limited Adaptive Histogram Equalization)
clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
master_gray=clahe.apply(master_image_gray)
test_gray=clahe.apply(test_image_gray)

In [ ]:
def gradient_magnitude(image):
    sobelx = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=3)
    magnitude=cv2.magnitude(sobelx, sobely)
    magnitude=cv2.convertScaleAbs(magnitude)
    return magnitude

In [ ]:
master_grad=gradient_magnitude(master_blur)
test_grad=gradient_magnitude(test_blur)

In [ ]:
diff=cv2.absdiff(master_grad,test_grad)
_,thresh=cv2.threshold(diff,30,255,cv2.THRESH_BINARY)
thresh=cv2.medianBlur(thresh,5)


In [ ]:
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)

In [ ]:
contours,_=cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
output=test_image.copy()
for i in contours:
    area=cv2.contourArea(i)
    x,y,w,h=cv2.boundingRect(i)
    aspect_ratio=w/float(h)
    if area>200:
        if 0.2 < aspect_ratio < 5.0:
            cv2.rectangle(output, (x, y), (x + w, y + h), (0, 255, 0), 2)

In [ ]:
fig,(ax1,ax2)=plt.subplots(1,2,figsize=(10,5))
ax1.imshow(master_image)
ax2.imshow(output)


**Gradient difference method selected since the images are tested in a controlled environment**

**Template Matching**

In [ ]:
master_path=r"C:\ALL PROJECTS\MCCB-Defect-Detection\images\mccb(R).png"
test_path=r"C:\ALL PROJECTS\MCCB-Defect-Detection\images\mccb(T).png"

In [ ]:
master_image=cv2.imread(master_path)
test_image=cv2.imread(test_path)

In [ ]:
print(master_image.shape,test_image.shape)

In [ ]:
#Grayscale
master_image_gray=cv2.cvtColor(master_image,cv2.COLOR_BGR2GRAY)
test_image_gray=cv2.cvtColor(test_image,cv2.COLOR_BGR2GRAY)

In [ ]:
#CLAHE
clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
master_image_clahe=clahe.apply(master_image_gray)
test_image_clahe=clahe.apply(test_image_gray)

**TEMPLATE MATCHING discarded because we were not able to find ROI that was consistent in all images**

In [ ]:
img=cv2.imread(r"D:\MCCB-Defect-Detection\cropped_master_imaeges\cropped_masterXT14P_mccb.png")
print(img.shape)

In [1]:
# Run this once to check
import cv2
print(cv2.__version__)
try:
    s = cv2.SIFT_create()
    print("SIFT available ✅")
except:
    
    print("SIFT not available — install opencv-contrib-python")

4.6.0
SIFT available ✅
